# 04 — Knowledge Base Retrieval Evaluation

**Task 08/08 deliverable** — Retrieval notebook + `retrieval_eval.json`.

This notebook:
1. Loads the EVVO Knowledge Base (KB v1.1, 91 entries across 11 categories).
2. Runs 10 gold queries covering realistic pentest-report-review scenarios.
3. Computes precision@5 and recall for each query.
4. Writes results to `data/kb/retrieval_eval.json`.

**Retrieval method**: metadata-based filtering (taxonomy_codes + domain + tags).
Vector index (embeddings + FAISS) will be added in task 13/08.

**Gold query categories covered** (per Definition of Done PDF §3.3):
- severity_logic (Q001, Q009)
- validation_rules (Q010)
- remediation_patterns (Q003, Q004)
- report_style (Q005)
- sop_methodology (Q008)
- client_qa_patterns (Q006, Q007)
- evidence/classification (Q002)


In [8]:
import sys, json
from pathlib import Path
from datetime import datetime, timezone

# Repo setup — adjust paths if running outside the repo
REPO_ROOT = Path(r'D:\evvo-slm-harness')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from harness.kb.loader import KBLoader
from harness.kb.retriever import KBRetriever

KB_ROOT = REPO_ROOT / 'data' / 'kb'
EVAL_OUT = KB_ROOT / 'retrieval_eval.json'

print(f'Repo root: {REPO_ROOT}')
print(f'KB root:   {KB_ROOT}')
print(f'Eval out:  {EVAL_OUT}')

Repo root: D:\evvo-slm-harness
KB root:   D:\evvo-slm-harness\data\kb
Eval out:  D:\evvo-slm-harness\data\kb\retrieval_eval.json


In [9]:
loader = KBLoader(kb_root=str(KB_ROOT))
entries = loader.load_all()
print(f'Loaded: {len(entries)} entries, {len(loader.load_errors)} errors')
if loader.load_errors:
    for e in loader.load_errors[:5]:
        print(f'  ERROR: {e}')

from collections import Counter
cat_counts = Counter(e.category for e in entries)
print('\nEntries by category:')
for cat, n in sorted(cat_counts.items()):
    print(f'  {cat:30s} {n}')

# Verify metadata fields are populated on a sample
sample = entries[0]
print(f'\nSample metadata ({sample.kb_id}):')
for f in ['document_type', 'section', 'vulnerability_type', 'effective_date', 'access_scope', 'source_id']:
    print(f'  {f}: {getattr(sample, f)!r}')

Loaded: 91 entries, 0 errors

Entries by category:
  classification_criteria        6
  consistency_rules              4
  escalation_rules               6
  evidence_standards             6
  governance_rules               4
  remediation_guidance           12
  severity_guidance              7
  sop                            19
  taxonomy_definitions           4
  validation_requirements        6
  writing_guidelines             17

Sample metadata (KB-CLASS-001):
  document_type: 'policy_document'
  section: 'classification_criteria:confirmed_vulnerability'
  vulnerability_type: 'general'
  effective_date: '2026-08-15'
  access_scope: 'internal'
  source_id: 'POLICY-review_taxonomy'


## Gold Queries

10 hand-crafted queries covering all 6 KB coverage categories.
Each query specifies:
- `taxonomy_codes`: taxonomy codes to retrieve (None = skip taxonomy filter)
- `domain`: finding domain filter ('all' = no domain filter)
- `tags`: tag filter (None = skip tag filter — mirrors production usage)
- `categories`: direct category filter (used for STYLE/QA/METH queries where
  no taxonomy code maps to the target category)
- `expected_kb_ids`: minimum set that MUST be retrieved (recall check)
- `relevant_kb_ids`: broader set considered relevant (precision@5 denominator)

**Note on retrieval design**: production code (orchestrator.py) calls
`retrieve_for_review(taxonomy_codes=[9 codes], domain=finding_domain)` — no tags.
These gold queries mirror that pattern, only using `categories` for cases
where the target KB entries live in a category not reachable via taxonomy codes
(writing_guidelines, sop). This is a known v1 limitation — task 13/08 will
introduce vector retrieval to bridge this gap.

In [10]:
import json as _json
GOLD_QUERIES = _json.loads(r'''
[
  {
    "query_id": "Q001",
    "scenario": "Review a Critical-severity finding with CVSS 9.8 — needs severity + consistency rules",
    "taxonomy_codes": [
      "SEV",
      "CVSS",
      "CONS"
    ],
    "domain": "all",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-SEV-001",
      "KB-CONS-001"
    ],
    "relevant_kb_ids": [
      "KB-SEV-001",
      "KB-SEV-006",
      "KB-SEV-007",
      "KB-CONS-001",
      "KB-VAL-002"
    ]
  },
  {
    "query_id": "Q002",
    "scenario": "Finding has only scanner output as evidence — classify as false positive or potential?",
    "taxonomy_codes": [
      "EVID",
      "CLASS"
    ],
    "domain": "all",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-EVID-004",
      "KB-CLASS-004"
    ],
    "relevant_kb_ids": [
      "KB-EVID-001",
      "KB-EVID-004",
      "KB-CLASS-002",
      "KB-CLASS-004",
      "KB-VAL-001"
    ]
  },
  {
    "query_id": "Q003",
    "scenario": "Finding has hardcoded credentials in mobile APK — needs remediation template",
    "taxonomy_codes": [
      "REC"
    ],
    "domain": "android_application",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-REC-TPL-001",
      "KB-REC-TPL-006"
    ],
    "relevant_kb_ids": [
      "KB-REC-TPL-001",
      "KB-REC-TPL-004",
      "KB-REC-TPL-006",
      "KB-REC-005",
      "KB-REC-006"
    ]
  },
  {
    "query_id": "Q004",
    "scenario": "Finding uses HS256 JWT — needs migration template + crypto guidance",
    "taxonomy_codes": [
      "REC",
      "SEV"
    ],
    "domain": "all",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-REC-TPL-002"
    ],
    "relevant_kb_ids": [
      "KB-REC-TPL-002",
      "KB-SEV-003",
      "KB-CONS-001",
      "KB-REC-001",
      "KB-REC-002"
    ]
  },
  {
    "query_id": "Q005",
    "scenario": "Review a finding's report style — check heading hierarchy and metadata table",
    "taxonomy_codes": null,
    "domain": "all",
    "tags": null,
    "categories": [
      "writing_guidelines"
    ],
    "expected_kb_ids": [
      "KB-STYLE-001",
      "KB-STYLE-004"
    ],
    "relevant_kb_ids": [
      "KB-STYLE-001",
      "KB-STYLE-002",
      "KB-STYLE-003",
      "KB-STYLE-004",
      "KB-STYLE-005"
    ]
  },
  {
    "query_id": "Q006",
    "scenario": "Client asks about a finding marked (Potential) — answer conditionally",
    "taxonomy_codes": null,
    "domain": "all",
    "tags": null,
    "categories": [
      "writing_guidelines",
      "classification_criteria"
    ],
    "expected_kb_ids": [
      "KB-QA-003",
      "KB-CLASS-002"
    ],
    "relevant_kb_ids": [
      "KB-QA-003",
      "KB-CLASS-002",
      "KB-QA-001",
      "KB-QA-002",
      "KB-VAL-001"
    ]
  },
  {
    "query_id": "Q007",
    "scenario": "Client asks about scope-out asset — must refuse",
    "taxonomy_codes": null,
    "domain": "all",
    "tags": null,
    "categories": [
      "writing_guidelines"
    ],
    "expected_kb_ids": [
      "KB-QA-005"
    ],
    "relevant_kb_ids": [
      "KB-QA-005",
      "KB-QA-002",
      "KB-QA-003",
      "KB-STYLE-003"
    ]
  },
  {
    "query_id": "Q008",
    "scenario": "Review a finding — needs full pentest methodology context (NIST/OWASP)",
    "taxonomy_codes": null,
    "domain": "all",
    "tags": null,
    "categories": [
      "sop"
    ],
    "expected_kb_ids": [
      "KB-SOP-METH-001",
      "KB-SOP-METH-003"
    ],
    "relevant_kb_ids": [
      "KB-SOP-METH-001",
      "KB-SOP-METH-002",
      "KB-SOP-METH-003",
      "KB-SOP-METH-004",
      "KB-SOP-METH-005"
    ]
  },
  {
    "query_id": "Q009",
    "scenario": "Finding has severity High but evidence insufficient — escalate",
    "taxonomy_codes": [
      "CONF",
      "EVID",
      "SEV"
    ],
    "domain": "all",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-ESC-001",
      "KB-SEV-006"
    ],
    "relevant_kb_ids": [
      "KB-ESC-001",
      "KB-ESC-002",
      "KB-ESC-004",
      "KB-SEV-006",
      "KB-EVID-002"
    ]
  },
  {
    "query_id": "Q010",
    "scenario": "Validate finding schema — check 5-part structure + CVSS + CWE format",
    "taxonomy_codes": [
      "COMP",
      "CVSS",
      "CWE"
    ],
    "domain": "all",
    "tags": null,
    "categories": null,
    "expected_kb_ids": [
      "KB-VAL-001",
      "KB-VAL-002",
      "KB-VAL-003"
    ],
    "relevant_kb_ids": [
      "KB-VAL-001",
      "KB-VAL-002",
      "KB-VAL-003",
      "KB-VAL-004",
      "KB-VAL-005",
      "KB-VAL-006"
    ]
  }
]
''')

print(f'Loaded {len(GOLD_QUERIES)} gold queries')
for q in GOLD_QUERIES:
    print(f"  {q['query_id']}: {q['scenario'][:80]}")

Loaded 10 gold queries
  Q001: Review a Critical-severity finding with CVSS 9.8 — needs severity + consistency 
  Q002: Finding has only scanner output as evidence — classify as false positive or pote
  Q003: Finding has hardcoded credentials in mobile APK — needs remediation template
  Q004: Finding uses HS256 JWT — needs migration template + crypto guidance
  Q005: Review a finding's report style — check heading hierarchy and metadata table
  Q006: Client asks about a finding marked (Potential) — answer conditionally
  Q007: Client asks about scope-out asset — must refuse
  Q008: Review a finding — needs full pentest methodology context (NIST/OWASP)
  Q009: Finding has severity High but evidence insufficient — escalate
  Q010: Validate finding schema — check 5-part structure + CVSS + CWE format


## Run Retrieval + Compute Metrics

For each query:
1. Call `KBRetriever.retrieve_for_review()` with the query parameters.
2. Compute **recall** = |retrieved ∩ expected| / |expected|.
3. Compute **precision@5** = |top-5 retrieved ∩ relevant| / 5.
4. Record per-query results.

In [11]:
retriever = KBRetriever(kb_root=str(KB_ROOT))

def compute_metrics(retrieved_ids, expected_ids, relevant_ids, k=5):
    retrieved_set = set(retrieved_ids)
    expected_set = set(expected_ids)
    relevant_set = set(relevant_ids)
    
    # Recall: did we retrieve all expected IDs?
    if expected_set:
        recall = len(retrieved_set & expected_set) / len(expected_set)
    else:
        recall = 1.0
    
    # Precision@k: of the top-k retrieved, how many are relevant?
    top_k = retrieved_ids[:k]
    if top_k:
        precision_at_k = len(set(top_k) & relevant_set) / len(top_k)
    else:
        precision_at_k = 0.0
    
    missing = sorted(expected_set - retrieved_set)
    return recall, precision_at_k, missing

results = []
for q in GOLD_QUERIES:
    # Mirror production usage: skip None filters, treat 'all' domain as no filter
    domain = q['domain'] if q['domain'] and q['domain'] != 'all' else None
    tags = q.get('tags') or None
    taxonomy_codes = q.get('taxonomy_codes') or None
    categories = q.get('categories') or None
    
    res = retriever.retrieve_for_review(
        taxonomy_codes=taxonomy_codes,
        domain=domain,
        tags=tags,
        categories=categories,
    )
    retrieved_ids = [e.kb_id for e in res.entries]
    recall, p_at_5, missing = compute_metrics(
        retrieved_ids, q['expected_kb_ids'], q['relevant_kb_ids']
    )
    
    # Collect which expected IDs were found vs missing
    expected_found = [kid for kid in q['expected_kb_ids'] if kid in retrieved_ids]
    
    results.append({
        'query_id': q['query_id'],
        'scenario': q['scenario'],
        'query_params': {
            'taxonomy_codes': taxonomy_codes,
            'domain': q['domain'],
            'tags': tags,
            'categories': categories,
        },
        'retrieved_count': len(retrieved_ids),
        'retrieved_kb_ids': retrieved_ids,
        'expected_kb_ids': q['expected_kb_ids'],
        'expected_found': expected_found,
        'expected_missing': missing,
        'relevant_kb_ids': q['relevant_kb_ids'],
        'recall': round(recall, 4),
        'precision_at_5': round(p_at_5, 4),
    })
    
    status = 'OK' if recall == 1.0 else 'FAIL'
    print(f"  [{status}] {q['query_id']}: recall={recall:.3f}, p@5={p_at_5:.3f}, retrieved={len(retrieved_ids)}")
    if missing:
        print(f"         MISSING: {missing}")

  [OK] Q001: recall=1.000, p@5=0.600, retrieved=11
  [OK] Q002: recall=1.000, p@5=0.400, retrieved=12
  [OK] Q003: recall=1.000, p@5=0.200, retrieved=10
  [OK] Q004: recall=1.000, p@5=0.200, retrieved=19
  [OK] Q005: recall=1.000, p@5=0.000, retrieved=17
  [OK] Q006: recall=1.000, p@5=0.400, retrieved=23
  [OK] Q007: recall=1.000, p@5=0.600, retrieved=17
  [OK] Q008: recall=1.000, p@5=0.200, retrieved=19
  [OK] Q009: recall=1.000, p@5=0.600, retrieved=19
  [OK] Q010: recall=1.000, p@5=0.400, retrieved=13


## Aggregate + Write `retrieval_eval.json`

Compute aggregate metrics (mean recall, mean precision@5, pass rate) and write the eval JSON.

In [12]:
n = len(results)
mean_recall = sum(r['recall'] for r in results) / n if n else 0
mean_p5 = sum(r['precision_at_5'] for r in results) / n if n else 0
n_pass = sum(1 for r in results if r['recall'] == 1.0)
n_fail = n - n_pass

eval_output = {
    'eval_version': '1.0',
    'kb_version': '1.1',
    'evaluated_at': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'retrieval_method': 'metadata_filter (taxonomy + domain + tags) — vector index planned for task 13/08',
    'total_queries': n,
    'aggregate_metrics': {
        'mean_recall': round(mean_recall, 4),
        'mean_precision_at_5': round(mean_p5, 4),
        'pass_rate': round(n_pass / n, 4) if n else 0,
        'queries_passed': n_pass,
        'queries_failed': n_fail,
    },
    'coverage_categories': {
        'severity_logic': ['Q001', 'Q009'],
        'validation_rules': ['Q010'],
        'remediation_patterns': ['Q003', 'Q004'],
        'report_style': ['Q005'],
        'sop_methodology': ['Q008'],
        'client_qa_patterns': ['Q006', 'Q007'],
        'evidence_classification': ['Q002'],
    },
    'queries': results,
}

EVAL_OUT.parent.mkdir(parents=True, exist_ok=True)
with open(EVAL_OUT, 'w', encoding='utf-8') as f:
    json.dump(eval_output, f, indent=2, ensure_ascii=False)
    f.write('\n')

print(f'Wrote {EVAL_OUT}')
print(f'\n=== Aggregate Metrics ===')
print(f'  Mean recall:        {mean_recall:.4f}')
print(f'  Mean precision@5:   {mean_p5:.4f}')
print(f'  Pass rate:          {n_pass}/{n} ({n_pass/n*100:.1f}%)')
print(f'  Queries failed:     {n_fail}')

Wrote D:\evvo-slm-harness\data\kb\retrieval_eval.json

=== Aggregate Metrics ===
  Mean recall:        1.0000
  Mean precision@5:   0.3600
  Pass rate:          10/10 (100.0%)
  Queries failed:     0


## Per-Query Details

Display each query's retrieved IDs vs expected IDs for manual inspection.

In [13]:
for r in results:
    print(f"\n{r['query_id']}: {r['scenario']}")
    print(f"  Query:   tax={r['query_params']['taxonomy_codes']}, domain={r['query_params']['domain']}, tags={r['query_params']['tags']}")
    print(f"  Recall:  {r['recall']}  (found {len(r['expected_found'])}/{len(r['expected_kb_ids'])} expected)")
    print(f"  P@5:     {r['precision_at_5']}")
    print(f"  Retrieved ({r['retrieved_count']}):")
    for kid in r['retrieved_kb_ids'][:10]:
        marker = ' [EXPECTED]' if kid in r['expected_kb_ids'] else (' [RELEVANT]' if kid in r['relevant_kb_ids'] else '')
        print(f"    - {kid}{marker}")
    if len(r['retrieved_kb_ids']) > 10:
        print(f"    ... and {len(r['retrieved_kb_ids'])-10} more")
    if r['expected_missing']:
        print(f"  MISSING: {r['expected_missing']}")


Q001: Review a Critical-severity finding with CVSS 9.8 — needs severity + consistency rules
  Query:   tax=['SEV', 'CVSS', 'CONS'], domain=all, tags=None
  Recall:  1.0  (found 2/2 expected)
  P@5:     0.6
  Retrieved (11):
    - KB-CONS-002
    - KB-SEV-001 [EXPECTED]
    - KB-SEV-002
    - KB-SEV-006 [RELEVANT]
    - KB-CONS-001 [EXPECTED]
    - KB-CONS-003
    - KB-CONS-004
    - KB-SEV-003
    - KB-SEV-004
    - KB-SEV-007 [RELEVANT]
    ... and 1 more

Q002: Finding has only scanner output as evidence — classify as false positive or potential?
  Query:   tax=['EVID', 'CLASS'], domain=all, tags=None
  Recall:  1.0  (found 2/2 expected)
  P@5:     0.4
  Retrieved (12):
    - KB-CLASS-001
    - KB-CLASS-002 [RELEVANT]
    - KB-CLASS-004 [EXPECTED]
    - KB-CLASS-005
    - KB-EVID-002
    - KB-EVID-003
    - KB-EVID-004 [EXPECTED]
    - KB-EVID-006
    - KB-CLASS-006
    - KB-EVID-005
    ... and 2 more

Q003: Finding has hardcoded credentials in mobile APK — needs remediation templa

## Summary

This notebook is re-runnable. After any KB change (adding/removing entries),
re-execute this notebook to regenerate `data/kb/retrieval_eval.json`.

**Next step (task 13/08)**: upgrade the retriever to support vector embeddings
(sentence-transformers + FAISS) and re-evaluate. Compare precision/recall before/after
to measure the improvement from semantic retrieval.